# Solvent Embedder: Valence-aware Embedding Fine-tuning

## Force is Oblivion / Hegemonikón Research (Peira)

**目的**: all-mpnet-base-v2 に Valence 弁別力を付与する。

**理論的根拠** (THEORY.md §4-6):
- ker(G) = {Scale, Valence}
- FIM stiff modes = 2 (3座標 → 2方向に圧縮)
- → Valence 方向を cos similarity 空間に追加すれば stiff modes >= 3

**手法**: AllNLI triplet (anchor, entailment, contradiction) → TripletLoss

**評価**:
- STS-B Spearman (汎用性の保持: 目標 >95%)
- Valence 弁別ギャップ (対立ペア vs 類似ペア cos差: 目標 >0.1)

**GPU**: L4 で ~30分 (50K samples, 3 epochs)

In [ ]:
# Cell 1: 環境
!nvidia-smi
!pip install -q sentence-transformers datasets

In [ ]:
# Cell 2: モデル + データ読込
import torch
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
import json

BASE_MODEL = 'sentence-transformers/all-mpnet-base-v2'
MAX_SAMPLES = 50000
EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5
TRIPLET_MARGIN = 0.3

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model = SentenceTransformer(BASE_MODEL, device=device)
print(f'Model dim: {model.get_sentence_embedding_dimension()}')

# AllNLI triplet
train_ds = load_dataset('sentence-transformers/all-nli', 'triplet',
                        split=f'train[:{MAX_SAMPLES}]')
print(f'Triplets: {len(train_ds)}')

# STS-B eval
sts_ds = load_dataset('sentence-transformers/stsb', split='test')
sts_eval = EmbeddingSimilarityEvaluator(
    sentences1=sts_ds['sentence1'],
    sentences2=sts_ds['sentence2'],
    scores=[s / 5.0 for s in sts_ds['score']],
    name='stsb-test',
)

In [ ]:
# Cell 3: Valence 弁別テスト関数
def valence_test(model):
    valence_pairs = [
        ('この設計は美しく、効率的だ', 'この設計は醜く、非効率的だ'),
        ('実験は成功し、仮説が支持された', '実験は失敗し、仮説が棄却された'),
        ('このアプローチは問題を解決する', 'このアプローチは新たな問題を生む'),
        ('コードの品質は高い', 'コードの品質は低い'),
        ('This approach is elegant and effective', 'This approach is clumsy and useless'),
    ]
    similar_pairs = [
        ('この設計は美しく、効率的だ', 'この設計は優雅で、パフォーマンスが良い'),
        ('実験は成功した', 'テストは pass した'),
        ('コードの品質は高い', 'コードはよく書かれている'),
        ('This is a good solution', 'This is an excellent approach'),
    ]
    def cos(a, b):
        e = model.encode([a, b], convert_to_tensor=True)
        return torch.nn.functional.cosine_similarity(e[0:1], e[1:2]).item()
    
    v_sims = [cos(a, b) for a, b in valence_pairs]
    s_sims = [cos(a, b) for a, b in similar_pairs]
    mv, ms = sum(v_sims)/len(v_sims), sum(s_sims)/len(s_sims)
    gap = ms - mv
    print(f'  対立ペア cos: {mv:.4f}')
    print(f'  類似ペア cos: {ms:.4f}')
    print(f'  弁別ギャップ: {gap:.4f} (目標>0.1)')
    return {'mean_valence': mv, 'mean_similar': ms, 'gap': gap}

# Baseline
print('=== Baseline ===')
bl_sts = sts_eval(model)
bl_spearman = bl_sts['stsb-test_spearman_cosine']
print(f'STS-B Spearman: {bl_spearman:.4f}')
bl_val = valence_test(model)

In [ ]:
# Cell 4: 訓練
triplet_loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=TRIPLET_MARGIN,
)

args = SentenceTransformerTrainingArguments(
    output_dir='/content/valence-mpnet',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    warmup_ratio=0.1,
    learning_rate=LR,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='stsb-test_spearman_cosine',
    save_total_limit=2,
)

trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=train_ds,
    loss=triplet_loss,
    evaluator=sts_eval,
)

trainer.train()

In [ ]:
# Cell 5: 最終評価
print('=== 最終評価 ===')
final_sts = sts_eval(model)
final_spearman = final_sts['stsb-test_spearman_cosine']
retention = final_spearman / bl_spearman * 100

print(f'STS-B baseline:  {bl_spearman:.4f}')
print(f'STS-B final:     {final_spearman:.4f}')
print(f'STS retention:   {retention:.1f}% (目標>95%)')
print()
ft_val = valence_test(model)

print(f'\n=== サマリー ===')
print(f'STS retention: {retention:.1f}%')
print(f'Valence gap baseline:  {bl_val["gap"]:.4f}')
print(f'Valence gap finetuned: {ft_val["gap"]:.4f}')
print(f'Gap 改善: {ft_val["gap"] - bl_val["gap"]:+.4f}')

# JSON 保存
meta = {
    'base_model': BASE_MODEL, 'epochs': EPOCHS,
    'max_samples': MAX_SAMPLES, 'lr': LR,
    'sts_baseline': bl_spearman, 'sts_final': final_spearman,
    'sts_retention_pct': retention,
    'baseline_valence_gap': bl_val['gap'],
    'finetuned_valence_gap': ft_val['gap'],
    'gap_improvement': ft_val['gap'] - bl_val['gap'],
}
with open('/content/solvent_results.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('\n結果保存: /content/solvent_results.json')

In [ ]:
# Cell 6: ダウンロード
from google.colab import files
files.download('/content/solvent_results.json')